In [15]:
import pandas as pd
import numpy as np

Load forecast and calculation of demand stats

In [9]:
forecast = pd.read_csv('../outputs/forecast_output.csv')
forecast['ds'] = pd.to_datetime(forecast['ds'])

last_train_date = '2012-04-13'
future_forecast = forecast[forecast['ds'] >= last_train_date].copy()
avg_weekly_demand = future_forecast['yhat'].mean()
demand_std = future_forecast['yhat'].std()
annual_demand = avg_weekly_demand * 52

print(f"Average Weekly Demand : {avg_weekly_demand:,.2f} units")
print(f"Demand Std Dev        : {demand_std:,.2f} units")
print(f"Annual Demand         : {annual_demand:,.2f} units")

Average Weekly Demand : 19,950.80 units
Demand Std Dev        : 4,985.91 units
Annual Demand         : 1,037,441.37 units


 EOQ calculation

In [10]:
ordering_cost = 500
unit_cost = 25
holding_cost = unit_cost * 0.25

eoq = np.sqrt((2 * annual_demand * ordering_cost) / holding_cost)
orders_per_year = annual_demand / eoq
cycle_time_weeks = 52 / orders_per_year
total_inventory_cost = (eoq / 2) * holding_cost + (annual_demand / eoq) * ordering_cost

print(f"EOQ                  : {eoq:,.2f} units")
print(f"Orders Per Year      : {orders_per_year:.2f}")
print(f"Cycle Time           : {cycle_time_weeks:.2f} weeks")
print(f"Total Inventory Cost : ${total_inventory_cost:,.2f}")

EOQ                  : 12,883.73 units
Orders Per Year      : 80.52
Cycle Time           : 0.65 weeks
Total Inventory Cost : $80,523.34


 Safety stock and ROP at multiple service levels

In [11]:
lead_time_weeks = 2
service_levels = {'90%': 1.28, '95%': 1.65, '99%': 2.33}

rows = []
for level, z in service_levels.items():
    safety_stock = z * demand_std * np.sqrt(lead_time_weeks)
    rop = (avg_weekly_demand * lead_time_weeks) + safety_stock
    rows.append({
        'Service Level': level,
        'Safety Stock (units)': round(safety_stock, 0),
        'Reorder Point (units)': round(rop, 0),
        'Z Score': z
    })

safety_df = pd.DataFrame(rows)
safety_df

,Service Level,Safety Stock (units),Reorder Point (units),Z Score
0,90%,9025.0,48927.0,1.28
1,95%,11634.0,51536.0,1.65
2,99%,16429.0,56331.0,2.33


 Cost reduction vs arbitrary policy

In [12]:
arbitrary_order_qty = avg_weekly_demand * 4
arbitrary_cost = (arbitrary_order_qty / 2) * holding_cost + (annual_demand / arbitrary_order_qty) * ordering_cost
cost_reduction_pct = ((arbitrary_cost - total_inventory_cost) / arbitrary_cost) * 100

comparison = pd.DataFrame({
    'Policy': ['Arbitrary (4-week order cycle)', 'EOQ Optimal'],
    'Order Quantity': [round(arbitrary_order_qty, 2), round(eoq, 2)],
    'Annual Cost ($)': [round(arbitrary_cost, 2), round(total_inventory_cost, 2)]
})

print(f"Cost Reduction: {cost_reduction_pct:.2f}%")
comparison

Cost Reduction: 68.53%


,Policy,Order Quantity,Annual Cost ($)
0,Arbitrary (4-week order cycle),79803.18,255884.94
1,EOQ Optimal,12883.73,80523.34


Export outputs

In [13]:
summary = pd.DataFrame({
    'Metric': ['Avg Weekly Demand', 'Annual Demand', 'EOQ',
               'Orders Per Year', 'Total Inventory Cost', 'Cost Reduction %'],
    'Value': [round(avg_weekly_demand, 2), round(annual_demand, 2),
              round(eoq, 2), round(orders_per_year, 2),
              round(total_inventory_cost, 2), round(cost_reduction_pct, 2)]
})

summary.to_csv('../outputs/inventory_metrics.csv', index=False)
safety_df.to_csv('../outputs/safety_stock_analysis.csv', index=False)

print("Saved inventory_metrics.csv and safety_stock_analysis.csv")

Saved inventory_metrics.csv and safety_stock_analysis.csv
